In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, roc_curve, auc, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import polars as pl

# Carrega dados
df_final = pl.read_parquet('df_final_ml_features.parquet')

# Converte colunas para tipos categóricos apropriados
df_final = df_final.with_columns(
    pl.col("Route_Cluster").cast(pl.Utf8).cast(pl.Categorical),
    pl.col("Day_of_week").cast(pl.Categorical),
    pl.col("Season").cast(pl.Utf8).cast(pl.Categorical),
    pl.col("Airline").cast(pl.Categorical),
    pl.col("Delayed").cast(pl.Int8)
)

print(f"✓ Dados carregados com sucesso!")
print(f"  Forma: {df_final.shape}")
print(f"  Colunas: {df_final.columns}")
print(f"\nTipos de dados:")
print(df_final.schema)

In [ ]:
# Prepara features e alvo usando codificação one-hot
df_pandas = df_final.to_pandas()

# Codifica one-hot variáveis categóricas
categorical_features = ['Day_of_week', 'Season', 'Airline', 'Route_Cluster']
df_encoded = pd.get_dummies(df_pandas, columns=categorical_features, drop_first=False)

# Separa features e alvo
X = df_encoded.drop('Delayed', axis=1)
y = df_encoded['Delayed']

feature_cols = X.columns.tolist()

print(f"✓ Features preparadas com codificação one-hot!")
print(f"  Forma das features: {X.shape}")
print(f"  Forma do alvo: {y.shape}")
print(f"  Número de features: {len(feature_cols)}")
print(f"\nDistribuição das classes:")
print(y.value_counts().sort_index())

In [ ]:
# Divisão treino-teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"✓ Divisão treino-teste concluída!")
print(f"  Conjunto de treinamento: {X_train.shape}")
print(f"  Conjunto de teste: {X_test.shape}")
print(f"\n  Distribuição de classes no treino:")
print(y_train.value_counts().sort_index())
print(f"\n  Distribuição de classes no teste:")
print(y_test.value_counts().sort_index())

In [ ]:
# Treina modelo XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    scale_pos_weight=sum(y_train == 0) / sum(y_train == 1)  # Trata desbalanceamento de classes
)

print("Treinando modelo XGBoost...")
xgb_model.fit(X_train, y_train)
print(f"✓ Modelo treinado com sucesso!")
print(f"\nParâmetros do modelo:")
print(f"  n_estimators: {xgb_model.n_estimators}")
print(f"  max_depth: {xgb_model.max_depth}")
print(f"  learning_rate: {xgb_model.learning_rate}")
print(f"  scale_pos_weight: {xgb_model.scale_pos_weight}")

In [ ]:
# Faz predições
y_pred_train = xgb_model.predict(X_train)
y_pred_test = xgb_model.predict(X_test)
y_pred_proba_train = xgb_model.predict_proba(X_train)[:, 1]
y_pred_proba_test = xgb_model.predict_proba(X_test)[:, 1]

# Calcula métricas para conjunto de treinamento
from sklearn.metrics import roc_auc_score
train_accuracy = accuracy_score(y_train, y_pred_train)
train_recall = recall_score(y_train, y_pred_train)
train_auc = roc_auc_score(y_train, y_pred_proba_train)

# Calcula métricas para conjunto de teste
test_accuracy = accuracy_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)
test_auc = roc_auc_score(y_test, y_pred_proba_test)

print("✓ Predições realizadas!")
print(f"\nMétricas do Conjunto de Treinamento:")
print(f"  Acurácia: {train_accuracy:.4f}")
print(f"  Recall (Sensibilidade): {train_recall:.4f}")
print(f"  AUC: {train_auc:.4f}")

print(f"\nMétricas do Conjunto de Teste:")
print(f"  Acurácia: {test_accuracy:.4f}")
print(f"  Recall (Sensibilidade): {test_recall:.4f}")
print(f"  AUC: {test_auc:.4f}")

In [ ]:
# Matriz de confusão para conjunto de teste
cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
plt.figure(figsize=(8, 6))
disp.plot()
plt.xlabel('Predito')
plt.ylabel('Observado')
plt.title('Matriz de Confusão - XGBoost Conjunto de Teste')
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Métricas detalhadas
specificity = recall_score(y_test, y_pred_test, pos_label=0)
sensitivity = recall_score(y_test, y_pred_test, pos_label=1)

metrics_df = pd.DataFrame({
    'Métrica': ['Acurácia', 'Sensibilidade (Recall)', 'Especificidade'],
    'Conjunto de Teste': [test_accuracy, sensitivity, specificity]
})

print("\nMétricas Detalhadas:")
print(metrics_df.to_string(index=False))

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_test)
roc_auc = auc(fpr, tpr)

# Coeficiente GINI
gini = (roc_auc - 0.5) / 0.5

plt.figure(figsize=(12, 8))
plt.plot(fpr, tpr, marker='o', color='darkorchid', markersize=8, linewidth=2.5, label='Curva ROC')
plt.plot(fpr, fpr, color='gray', linestyle='dashed', linewidth=2, label='Classificador Aleatório')
plt.title(f'Curva ROC - XGBoost | AUC: {roc_auc:.4f} | GINI: {gini:.4f}', fontsize=16)
plt.xlabel('Taxa de Falso Positivo (1 - Especificidade)', fontsize=14)
plt.ylabel('Taxa de Verdadeiro Positivo (Sensibilidade)', fontsize=14)
plt.xticks(np.arange(0, 1.1, 0.2), fontsize=12)
plt.yticks(np.arange(0, 1.1, 0.2), fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAnálise ROC:")
print(f"  Pontuação AUC: {roc_auc:.4f}")
print(f"  Coeficiente GINI: {gini:.4f}")

In [ ]:
# Importância das features
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importância': xgb_model.feature_importances_
}).sort_values('Importância', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importância'], color='steelblue')
plt.xlabel('Pontuação de Importância', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Importância das Features - XGBoost', fontsize=14)
plt.tight_layout()
plt.show()

print("\nTop 10 Features Mais Importantes:")
print(feature_importance.head(10).to_string(index=False))